In [1]:
import torch

In [2]:
matrix = torch.rand(3, 3)
print("\nSample Tensor Matrix:\n", matrix)


Sample Tensor Matrix:
 tensor([[0.4255, 0.2554, 0.1660],
        [0.9965, 0.9159, 0.4124],
        [0.7668, 0.3977, 0.9628]])


In [3]:
torch.cuda.is_available()

False

In [4]:
cpu_tensor = torch.randn(2, 3, device='cpu')

# Check basic attributes
print("Device:", cpu_tensor.device)          # Output: cpu
print("Data Type:", cpu_tensor.dtype)        # Output: torch.float32


Device: cpu
Data Type: torch.float32


In [5]:
# Returns the number of logical or physical CPU sockets/devices recognized
print("CPU Device Count:", torch.cpu.device_count()) 

# Check the engine capabilities detected at runtime (AVX2, AVX512, etc.)
print("CPU Capabilities:", torch.cpu.get_capabilities())


CPU Device Count: 1
CPU Capabilities: {'pmull': True, 'sha2': True, 'aes': True, 'fhm': True, 'atomics': True, 'sve_bf16': False, 'architecture': 'arm64', 'cpu_name': 'Unknown', 'i8mm': False, 'rdm': True, 'num_sockets': 1, 'l2_cache_size': 262144, 'num_physical_cores': 10, 'num_logical_cores': 10, 'l1d_cache_size': 32768, 'sha1': True, 'sme2': False, 'sme': False, 'sve2': False, 'bf16': True, 'sve': False, 'neon': True, 'crc32': True, 'fp16_arith': True, 'dot': True}


In [7]:
import torch.nn as nn

# Define a simple compute sequence
class LinearBlock(nn.Module):
    def __init__(self):
        super().__init__()
        self.linear = nn.Linear(10, 10)
        self.relu = nn.ReLU()
    def forward(self, x):
        return self.relu(self.linear(x))

model = LinearBlock()

# Compile the model specifically for the CPU architecture
optimized_model = torch.compile(model)

# Run a baseline prediction pass
dummy_input = torch.randn(1, 10)
output = optimized_model(dummy_input)

In [10]:
x = torch.ones(3, 3) * 2
y = torch.ones(3, 3) * 4

# Native CPU addition and matrix multiplication
z = x + y
matrix_product = torch.matmul(x, y)

print("Matrix Product Result:\n", matrix_product)

Matrix Product Result:
 tensor([[24., 24., 24.],
        [24., 24., 24.],
        [24., 24., 24.]])


In [11]:
# 1. Inspect default thread setup (usually maps to physical core count)
print(f"Default Intra-op Threads: {torch.get_num_threads()}")
print(f"Default Inter-op Threads: {torch.get_num_interop_threads()}\n")

# 2. Limit execution footprint to exactly 2 worker threads
# WARNING: Always invoke this at the top of your script before running tensors!
torch.set_num_threads(2)
torch.set_num_interop_threads(2)

print(f"Throttled Intra-op Threads: {torch.get_num_threads()}")
print(f"Throttled Inter-op Threads: {torch.get_num_interop_threads()}")

Default Intra-op Threads: 10
Default Inter-op Threads: 10

Throttled Intra-op Threads: 2
Throttled Inter-op Threads: 2


In [12]:
import time

# --- Setup large matrix data ---
matrix_a = torch.randn(4000, 4000)
matrix_b = torch.randn(4000, 4000)

# --- Test 1: Single Core Mode (Ideal for low background footprint) ---
torch.set_num_threads(1)
start_single = time.time()
result_single = torch.matmul(matrix_a, matrix_b)
end_single = time.time()
print(f"Single-Thread Execution Time: {end_single - start_single:.4f} seconds")

# --- Test 2: Multi Core Mode (Ideal for maximum dedicated training) ---
# Adjust this number to match half of your actual physical core count for efficiency
torch.set_num_threads(4) 
start_multi = time.time()
result_multi = torch.matmul(matrix_a, matrix_b)
end_multi = time.time()
print(f"Multi-Thread (4 Cores) Time:  {end_multi - start_multi:.4f} seconds")


Single-Thread Execution Time: 2.0936 seconds
Multi-Thread (4 Cores) Time:  0.5315 seconds


In [13]:
# Check exact system acceleration mappings (AVX2, AVX512, AMX, etc.)
hardware_capabilities = torch.cpu.get_capabilities()

print("Processor Vector Support Extensions:")
for capability, supported in hardware_capabilities.items():
    if supported and isinstance(supported, bool):
        print(f" - {capability.upper()}: Supported")


Processor Vector Support Extensions:
 - PMULL: Supported
 - SHA2: Supported
 - AES: Supported
 - FHM: Supported
 - ATOMICS: Supported
 - RDM: Supported
 - SHA1: Supported
 - BF16: Supported
 - NEON: Supported
 - CRC32: Supported
 - FP16_ARITH: Supported
 - DOT: Supported


In [14]:
import torch
import torch.nn as nn
import torch.optim as optim

# 1. Enforce your optimal thread settings at the entry point
torch.set_num_threads(4)

# 2. Define a clean baseline deep neural network
class BudgetClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(20, 64),
            nn.ReLU(),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, 2)  # Binary classification architecture output
        )
        
    def forward(self, x):
        return self.network(x)

# 3. Instantiate model, loss metrics, and tracking optimization algorithms
model = BudgetClassifier()
criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(), lr=0.01)

# 4. Synthesize artificial dataset tensors explicitly on host memory
dummy_features = torch.randn(128, 20, device='cpu')
dummy_labels = torch.randint(0, 2, (128,), device='cpu')

# 5. Execute an isolated performance training cycle pass
model.train()
optimizer.zero_grad()               # Clear mathematical gradients from previous cycles

# Computational execution graph step
predictions = model(dummy_features)
loss = criterion(predictions, dummy_labels)

# Compute loss derivatives back through the operational nodes
loss.backward()                     
optimizer.step()                    # Mutate weights across active structural matrix spaces

print(f"Training Cycle Completed Successfully.")
print(f"Current Batch Loss Footprint: {loss.item():.4f}")


Training Cycle Completed Successfully.
Current Batch Loss Footprint: 0.6969
